# FinDER: Vector RAG vs Graph RAG with Lance for both stores

This tutorial builds two parallel retrieval pipelines over the **FinDER** SEC 10-K Q&A dataset and answers the same questions through each:

1. **Vector RAG** — embed chunks, top-k similarity, augment the prompt. Backed by `LanceDBVectorStore`.
2. **Graph RAG** — entity extraction + linking, retrieve neighbors, augment the prompt. Backed by a small `LanceGraphStore` (a property-graph adapter on top of two LanceDB tables).
3. **Hybrid** — pass both vector hits and graph context into the answer synthesizer.

**Why Lance for both?** SEOCHO's embedded LadybugDB has no vector indexing, so a fully embedded RAG pipeline needs a vector backend. We use LanceDB. To keep the graph backend on the same storage layer, we ship `examples/lance_graph_store.py` — a tutorial-only adapter that implements the `seocho.store.graph.GraphStore` ABC over Lance tables. When upstream [lance-graph](https://github.com/lance-format/lance-graph/issues/91) lands, swapping is a one-import change.

**Dataset.** Set `FINDER_PATH` env var to your FinDER JSON (matches the format of `seocho.benchmarking.load_finder_cases`). If unset, this notebook uses a small synthetic 10-K excerpt at `examples/datasets/finder_tutorial_subset.json` so the cells run end-to-end without external data.

## Setup

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if (Path.cwd().name == "examples") else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

LLM_SPEC = os.environ.get("SEOCHO_LLM", "openai/gpt-4o-mini")
LLM_PROVIDER, LLM_MODEL = (LLM_SPEC.split("/", 1) + [""])[:2]
if not LLM_MODEL:
    LLM_PROVIDER, LLM_MODEL = "openai", LLM_SPEC

PROVIDER_KEY = {
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "kimi": "MOONSHOT_API_KEY",
    "grok": "XAI_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}.get(LLM_PROVIDER, "OPENAI_API_KEY")

if not os.environ.get(PROVIDER_KEY):
    raise RuntimeError(f"{PROVIDER_KEY} required for SEOCHO_LLM={LLM_SPEC}")
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is required for Tutorial 1's vector path "
        "(embeddings must use OpenAI even when SEOCHO_LLM points elsewhere)."
    )

FINDER_PATH = os.environ.get(
    "FINDER_PATH",
    str(ROOT / "examples" / "datasets" / "finder_tutorial_subset.json"),
)
WORKDIR = ROOT / ".seocho" / "finder_tutorial"
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"FinDER source:  {FINDER_PATH}")
print(f"Working dir:    {WORKDIR}")
print(f"LLM:            {LLM_PROVIDER}/{LLM_MODEL}")
print(f"Embeddings:     openai/text-embedding-3-small (fixed)")

In [ ]:
from seocho.benchmarking import load_finder_cases

cases = load_finder_cases(FINDER_PATH)
print(f"Loaded {len(cases)} FinDER cases")
for case in cases[:3]:
    print(f"  [{case.category}] {case.case_id}: {case.question}")

## Vector RAG path

In [ ]:
from seocho.store.vector import create_vector_store

vector_store = create_vector_store(
    kind="lancedb",
    uri=str(WORKDIR / "vec.lance"),
    table_name="finder_vec",
)

items = [
    {
        "id": case.case_id,
        "text": case.text,
        "metadata": {"category": case.category, "question": case.question},
    }
    for case in cases
]
added = vector_store.add_batch(items)
print(f"Indexed {added} chunks into LanceDB. Active rows: {vector_store.count()}")

In [ ]:
from seocho.store.llm import create_llm_backend

llm = create_llm_backend(provider=LLM_PROVIDER, model=LLM_MODEL)

ANSWER_SYSTEM = "Answer the question using only the supplied evidence. Be concise. If the evidence does not contain the answer, say so."

def answer_with_vector(question: str, *, k: int = 3) -> dict:
    t0 = time.perf_counter()
    hits = vector_store.search(question, limit=k)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    context = "\n\n".join(f"[{h.id}] {h.text}" for h in hits)
    user = f"Context:\n{context}\n\nQuestion: {question}"
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(context),
        "hit_count": len(hits),
    }

vec_demo = answer_with_vector(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {vec_demo['answer']}")
print(f"   ({vec_demo['retrieval_ms']:.1f} ms retrieve, {vec_demo['generation_ms']:.1f} ms generate, {vec_demo['hit_count']} hits)")

## Graph RAG path

We use a tiny ontology (Company/Person/FinancialMetric/Regulation, from `examples/datasets/fibo_base.jsonld`) and run extraction with the SEOCHO indexing engine. Writes go to a Lance-backed property graph rather than LadybugDB or Neo4j.

In [ ]:
from examples.lance_graph_store import LanceGraphStore
from seocho import Ontology, Seocho

ontology = Ontology.from_jsonld(ROOT / "examples" / "datasets" / "fibo_base.jsonld")
graph_store = LanceGraphStore(uri=str(WORKDIR / "graph.lance"))

client = Seocho(
    ontology=ontology,
    graph_store=graph_store,
    llm=llm,
    workspace_id="finder_tutorial",
)

for case in cases:
    client.add(case.text, metadata={"case_id": case.case_id, "category": case.category})

schema = graph_store.get_schema()
print(f"Graph contains {schema['node_count']} nodes and {schema['relationship_count']} relationships")
print(f"Labels: {schema['labels']}")
print(f"Rel types: {schema['relationship_types']}")

In [ ]:
# Lance has no Cypher engine, so we drive graph retrieval via the
# LanceGraphStore's tutorial methods directly: keyword_retrieve to seed
# relevant nodes from the question, then expand via neighbors() for
# one-hop context. This is the path that upstream lance-graph (#91) is
# expected to make smoother.
import re

STOPWORDS = {"what", "where", "who", "when", "how", "the", "a", "an", "is", "was", "were", "of", "in", "to", "for", "on", "and", "did", "during"}
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_.&\-]+")

def graph_context(question: str, *, seed_limit: int = 3, hop_limit: int = 5) -> str:
    tokens = [t for t in TOKEN_RE.findall(question) if t.lower() not in STOPWORDS]
    seeds: list[dict] = []
    for tok in tokens:
        seeds.extend(graph_store.keyword_retrieve(tok, limit=seed_limit))
    seen_ids: set[str] = set()
    facts: list[str] = []
    for seed in seeds:
        if seed["id"] in seen_ids:
            continue
        seen_ids.add(seed["id"])
        facts.append(f"{seed['label']}({seed['name']}) properties={seed['properties']}")
        for hop in graph_store.neighbors(seed["id"], limit=hop_limit):
            arrow = "->" if hop["direction"] == "out" else "<-"
            facts.append(
                f"{seed['name']} {arrow}[{hop['edge_type']}]{arrow} {hop['neighbor_label']}({hop['neighbor_name']})"
            )
    return "\n".join(facts)

demo_ctx = graph_context(cases[0].question)
print("Graph context for first question:")
print(demo_ctx or "(no matching nodes — try another question)")

In [ ]:
def answer_with_graph(question: str) -> dict:
    t0 = time.perf_counter()
    context = graph_context(question)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    if not context:
        return {
            "answer": "(no graph evidence)",
            "retrieval_ms": retrieval_ms,
            "generation_ms": 0.0,
            "context_chars": 0,
            "hit_count": 0,
        }
    user = f"Graph evidence:\n{context}\n\nQuestion: {question}"
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(context),
        "hit_count": context.count("\n") + 1,
    }

graph_demo = answer_with_graph(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {graph_demo['answer']}")
print(f"   ({graph_demo['retrieval_ms']:.1f} ms retrieve, {graph_demo['generation_ms']:.1f} ms generate, {graph_demo['hit_count']} graph facts)")

## Hybrid: vector + graph context

In [ ]:
def answer_hybrid(question: str, *, k: int = 3) -> dict:
    t0 = time.perf_counter()
    hits = vector_store.search(question, limit=k)
    vec_ctx = "\n\n".join(f"[{h.id}] {h.text}" for h in hits)
    g_ctx = graph_context(question)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    user = (
        f"Passages:\n{vec_ctx}\n\nGraph evidence:\n{g_ctx}\n\n"
        f"Question: {question}"
    )
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(vec_ctx) + len(g_ctx),
        "hit_count": len(hits) + (g_ctx.count("\n") + 1 if g_ctx else 0),
    }

hyb_demo = answer_hybrid(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {hyb_demo['answer']}")

## Side-by-side comparison

In [ ]:
from seocho.benchmarking import normalize_answer

def correct(answer: str, expected: str) -> bool:
    return normalize_answer(expected) in normalize_answer(answer)

rows = []
for case in cases:
    v = answer_with_vector(case.question)
    g = answer_with_graph(case.question)
    h = answer_hybrid(case.question)
    rows.append({
        "case_id": case.case_id,
        "category": case.category,
        "expected": case.expected_answer,
        "vector_answer": v["answer"],
        "vector_correct": correct(v["answer"], case.expected_answer),
        "vector_total_ms": v["retrieval_ms"] + v["generation_ms"],
        "graph_answer": g["answer"],
        "graph_correct": correct(g["answer"], case.expected_answer),
        "graph_total_ms": g["retrieval_ms"] + g["generation_ms"],
        "hybrid_answer": h["answer"],
        "hybrid_correct": correct(h["answer"], case.expected_answer),
        "hybrid_total_ms": h["retrieval_ms"] + h["generation_ms"],
    })

import pandas as pd
df = pd.DataFrame(rows)
summary = pd.DataFrame({
    "path": ["vector", "graph", "hybrid"],
    "correct_rate": [df["vector_correct"].mean(), df["graph_correct"].mean(), df["hybrid_correct"].mean()],
    "avg_total_ms": [df["vector_total_ms"].mean(), df["graph_total_ms"].mean(), df["hybrid_total_ms"].mean()],
})
print("Per-question detail:")
print(df[["case_id", "category", "vector_correct", "graph_correct", "hybrid_correct"]].to_string(index=False))
print("\nSummary:")
print(summary.to_string(index=False))

## Cleanup

Tutorial artifacts live under `.seocho/finder_tutorial/`. Delete that directory to reset.

In [ ]:
graph_store.close()
print(f"Artifacts: {WORKDIR}")